#Document-Based RAG Search Assistant

## 1. Environment setup

In [ ]:
# Run this cell in a fresh notebook environment if dependencies are not installed.
%pip install -U pydantic-settings python-dotenv langchain-core langchain-community langchain-openai langchain-google-genai langchain-groq langchain-chroma langchain-huggingface langchain-text-splitters chromadb sentence-transformers pypdf rich

## 2. Imports and logging

In [ ]:
import os
import re
import hashlib
import logging
from pathlib import Path
from typing import Iterable, Optional


from dotenv import load_dotenv
from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader


logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(name)s | %(message)s")
logger = logging.getLogger("doc_rag_assistant")
load_dotenv()

## 3. Configuration and directory initialization

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

LLM_PROVIDER = "gemini"  # openai, gemini, groq

MODEL_CONFIG = {
    "openai": {
        "model": "gpt-4o-mini",
        "temperature": 0,
        "api_key_env": "OPENAI_API_KEY",
    },
    "gemini": {
        "model": "gemini-3.8-flash",
        "temperature": 0,
        "api_key_env": "GEMINI_API_KEY",
    },
    "groq": {
        "model": "llama-3.3-70b-versatile",
        "temperature": 0,
        "api_key_env": "GROQ_API_KEY",
    },
}

print(f"Selected LLM Provider: {LLM_PROVIDER.upper()}")
print(f"Selected Model: {MODEL_CONFIG[LLM_PROVIDER]['model']}")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq


def get_llm(provider=None):

    provider = provider or LLM_PROVIDER

    if provider not in MODEL_CONFIG:
        raise ValueError(
            f"Unsupported provider: {provider}. "
            f"Choose from {list(MODEL_CONFIG.keys())}"
        )

    config = MODEL_CONFIG[provider]
    api_key_env = config["api_key_env"]

    api_key = os.getenv(api_key_env)

    # Google Colab Secret fallback
    if not api_key:
        try:
            from google.colab import userdata
            api_key = userdata.get(api_key_env)
        except Exception:
            api_key = None

    if not api_key:
        raise ValueError(
            f"Missing API key for {provider}. "
            f"Please configure {api_key_env} in .env "
            f"or Google Colab Secrets."
        )

    if provider == "openai":
        return ChatOpenAI(
            model=config["model"],
            temperature=config["temperature"],
            api_key=api_key,
        )

    elif provider == "gemini":
        return ChatGoogleGenerativeAI(
            model=config["model"],
            temperature=config["temperature"],
            google_api_key=api_key,
        )

    elif provider == "groq":
        return ChatGroq(
            model=config["model"],
            temperature=config["temperature"],
            api_key=api_key,
        )

In [ ]:
class Settings(BaseSettings):

    llm_provider: str = LLM_PROVIDER
    llm_model: str = MODEL_CONFIG[LLM_PROVIDER]["model"]

    embedding_model: str = Field(
        default="sentence-transformers/all-MiniLM-L6-v2",
        validation_alias="HF_EMBEDDING_MODEL"
    )

    chroma_persist_dir: str = "./vectorstore"
    collection_name: str = "rag_documents"
    data_dir: str = "./data"

    chunk_size: int = 1000
    chunk_overlap: int = 150

    relevance_threshold: float = 0.35
    top_k_results: int = 4

    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore"
    )

# INITIALIZE SETTINGS

settings = Settings()

# Create required directories
Path(settings.data_dir).mkdir(parents=True, exist_ok=True)
Path(settings.chroma_persist_dir).mkdir(parents=True, exist_ok=True)
Path("./logs").mkdir(parents=True, exist_ok=True)

# VALIDATION

if settings.chunk_size <= 0:
    raise ValueError("CHUNK_SIZE must be greater than zero")

if not 0 <= settings.chunk_overlap < settings.chunk_size:
    raise ValueError(
        "CHUNK_OVERLAP must be >= 0 and less than CHUNK_SIZE"
    )


# STATUS

print("=" * 45)
print("       DocRagAssistant Configuration")
print("=" * 45)

print(f"Provider       : {settings.llm_provider.upper()}")
print(f"Model         : {settings.llm_model}")
print(f"Embedding     : {settings.embedding_model}")
print(f"Documents     : {Path(settings.data_dir).resolve()}")
print("=" * 45)

## 4. Document loading, normalization, and chunking

In [ ]:
class DocumentProcessor:
    SUPPORTED_EXTENSIONS = {".pdf", ".txt", ".md"}


    def __init__(self, data_dir: str = settings.data_dir, chunk_size: int = settings.chunk_size,
                 chunk_overlap: int = settings.chunk_overlap):
        self.data_dir = Path(data_dir)
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        )


    @staticmethod
    def _clean_text(text: str) -> str:
        text = text.replace("\x00", " ")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()


    def load_files(self) -> list[Document]:
        if not self.data_dir.exists():
            logger.warning("Data folder does not exist: %s", self.data_dir)
            return []
        docs: list[Document] = []
        for path in sorted(self.data_dir.rglob("*")):
            if not path.is_file() or path.suffix.lower() not in self.SUPPORTED_EXTENSIONS:
                continue
            try:
                if path.suffix.lower() == ".pdf":
                    loaded = PyPDFLoader(str(path)).load()
                else:
                    text = path.read_text(encoding="utf-8", errors="replace")
                    loaded = [Document(page_content=text, metadata={"source": str(path), "page": 1})]
                for doc in loaded:
                    cleaned = self._clean_text(doc.page_content)
                    if cleaned:
                        metadata = dict(doc.metadata)
                        metadata["source"] = str(path)
                        docs.append(Document(page_content=cleaned, metadata=metadata))
            except Exception as exc:
                logger.exception("Skipping unreadable file %s: %s", path, exc)
        logger.info("Loaded %d non-empty document/page records", len(docs))
        return docs


    def process_and_chunk(self) -> list[Document]:
        docs = self.load_files()
        chunks = self.splitter.split_documents(docs) if docs else []
        for i, chunk in enumerate(chunks):
            chunk.metadata.setdefault("source", "unknown")
            chunk.metadata.setdefault("page", "N/A")
            chunk.metadata["chunk_index"] = i
        logger.info("Created %d chunks", len(chunks))
        return chunks


processor = DocumentProcessor()
# Preview loaded/chunked documents. This does not index them yet.
chunks = processor.process_and_chunk()
print(f"Chunks available: {len(chunks)}")
if chunks:
    print("Sample metadata:", chunks[0].metadata)
    print("Sample text:", chunks[0].page_content[:500])

## 5. Local embeddings and Chroma vector store

In [ ]:
class VectorStoreManager:
    def __init__(self, persist_dir: str = settings.chroma_persist_dir,
                 collection_name: str = settings.collection_name):
        self.persist_dir = persist_dir
        self.collection_name = collection_name
        self.embeddings = HuggingFaceEmbeddings(model_name=settings.embedding_model)
        self.vectorstore = Chroma(
            collection_name=collection_name,
            persist_directory=persist_dir,
            embedding_function=self.embeddings,
        )


    def index_documents(self, documents: list[Document], clean_existing: bool = False) -> bool:
        if not documents:
            logger.warning("No chunks supplied; skipping indexing")
            return False
        if clean_existing:
            # Recreate the persistent store cleanly before indexing.
            # Call this only when you intend to remove the current indexed corpus.
            self.vectorstore.delete_collection()
            self.vectorstore = Chroma(
                collection_name=self.collection_name,
                persist_directory=self.persist_dir,
                embedding_function=self.embeddings,
            )
        ids = []
        for doc in documents:
            digest = hashlib.sha256((doc.metadata.get("source", "") + "\n" + str(doc.metadata.get("chunk_index", len(ids))) + "\n" + doc.page_content).encode("utf-8")).hexdigest()
            ids.append(digest)
        try:
            self.vectorstore.add_documents(documents, ids=ids)
            logger.info("Indexed %d document chunks", len(documents))
            return True
        except Exception:
            logger.exception("Document indexing failed")
            return False


    def search(self, query: str, k: int = settings.top_k_results):
        if not query.strip():
            return []
        return self.vectorstore.similarity_search_with_score(query, k=k)


vector_manager = VectorStoreManager()
print("Vector store initialized at", settings.chroma_persist_dir)

## 6. Ingest/index documents

In [ ]:
def run_ingestion(clean_existing: bool = True) -> bool:
    """Load files under DATA_DIR, chunk, and index. clean_existing=True replaces the collection."""
    logger.info("Starting ingestion")
    new_chunks = DocumentProcessor().process_and_chunk()
    if not new_chunks:
        logger.warning("No valid chunks found. Add .pdf, .txt, or .md files to %s", settings.data_dir)
        return False
    manager = VectorStoreManager()
    success = manager.index_documents(new_chunks, clean_existing=clean_existing)
    if success:
        logger.info("Ingestion complete: %d chunk(s)", len(new_chunks))
    return success

# Run ingestion when ready. Keep clean_existing=True to replace prior corpus;
# set False to append/merge (duplicate IDs may be rejected by Chroma).
run_ingestion(clean_existing=True)

## 7. RAG engine with relevance guardrail and source metadata

In [ ]:
class RAGEngine:
    def __init__(self):
        self.store = VectorStoreManager()
        self.llm = get_llm()
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a document-grounded assistant. Answer using only the supplied context. "
                      "If the context does not contain the answer, say you cannot find it in the provided documents. "
                      "Do not invent facts. Cite source names and page numbers inline when possible."),
            ("human", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer based only on the context.")
        ])

    @staticmethod
    def _format_sources(retrieved) -> tuple[str, list[dict]]:
        pieces, sources = [], []
        for doc, distance in retrieved:
            source = Path(str(doc.metadata.get("source", "unknown"))).name
            page = doc.metadata.get("page", "N/A")
            # Chroma distance: lower is more similar. This is a rough display proxy, not calibrated confidence.
            distance_value = float(distance)
            confidence_proxy = max(0.0, 1.0 - distance_value)
            pieces.append(f"[Source: {source}, page/section: {page}]\n{doc.page_content}")
            sources.append({
                "source_name": source,
                "page": page,
                "relevance_distance": distance_value,
                "Confidence_score": confidence_proxy,
            })
        return "\n\n---\n\n".join(pieces), sources


    def answer_query(self, query: str) -> dict:
        query = query.strip()
        if not query:
            return {"answer": "Please enter a non-empty question.", "sources": [], "fallback_triggered": True}
        retrieved = self.store.search(query, k=settings.top_k_results)
        if not retrieved:
            return {"answer": "I couldn't find relevant information in the indexed documents.", "sources": [], "fallback_triggered": True}
        # Threshold applies to Chroma distance; lower distance indicates closer match.
        best_distance = min(float(score) for _, score in retrieved)
        if best_distance > settings.relevance_threshold:
            return {"answer": "I couldn't find sufficiently relevant information in the indexed documents to answer that.",
                    "sources": [], "fallback_triggered": True}
        context, sources = self._format_sources(retrieved)
        chain = self.prompt | self.llm
        response = chain.invoke({"context": context, "question": query})
        answer = response.content if hasattr(response, "content") else str(response)
        return {"answer": answer, "sources": sources, "fallback_triggered": False}

print("RAG engine class defined. Initialize it only after setting OPENAI_API_KEY.")

In [ ]:
# Check whether documents exist in the Chroma database

try:
    count = vector_manager.vectorstore._collection.count()

    print("Indexed document chunks:", count)

    if count == 0:
        print("WARNING: No documents are stored in the vector database.")
    else:
        print("SUCCESS: Documents exist in the vector database.")

except Exception as e:
    print("Error checking vector database:", e)

## 8. Query helper and citation display

In [ ]:
def ask(question: str):
    engine = RAGEngine()
    result = engine.answer_query(question)
    print("\nANSWER\n", result["answer"])
    if result["sources"]:
        print("\nSOURCES")
        for source in result["sources"]:
            print(f"- {source['source_name']} | page {source['page']} | distance={source['relevance_distance']:.4f} | proxy={source['Confidence_score']:.2%}")
    return result

ask("What does the document say about ...?")

In [ ]:
# Create a fresh connection to the Chroma vector database

fresh_manager = VectorStoreManager()

try:
    count = fresh_manager.vectorstore._collection.count()

    print("Indexed document chunks:", count)

    if count > 0:
        print("SUCCESS: Documents are indexed.")
    else:
        print("WARNING: Chroma collection exists but contains no documents.")

except Exception as e:
    print("Chroma database check failed:", e)

In [ ]:
test_question = "Summarize the main topics covered in my documents."

results = fresh_manager.search(test_question, k=3)

print("Retrieved documents:", len(results))

for i, (doc, distance) in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Distance:", round(float(distance), 4))
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print("\nRetrieved text:")
    print(doc.page_content[:1000])
    print("-" * 80)

In [ ]:
# TEMPORARY DIAGNOSTIC SETTING
settings.relevance_threshold = 1.7

print("Updated threshold:", settings.relevance_threshold)

## 9. Optional interactive terminal loop

In [ ]:
def interactive_cli():
    engine = RAGEngine()
    print("Document RAG assistant. Type 'exit' to quit or 'ingest' to rebuild the collection.")
    while True:
        try:
            query = input("\nQuestion> ").strip()
            if query.lower() in {"exit", "quit", "q"}:
                print("Goodbye!")
                break
            if query.lower() == "ingest":
                run_ingestion(clean_existing=True)
                continue
            if not query:
                continue
            result = engine.answer_query(query)
            print("\n" + result["answer"])
            for src in result["sources"]:
                print(f"Source: {src['source_name']} | page {src['page']} | distance {src['relevance_distance']:.4f}")
        except KeyboardInterrupt:
            print("\nStopped.")
            break
        except Exception as exc:
            logger.exception("Query failed: %s", exc)

# Uncomment to launch interactive mode:
interactive_cli()

## 10. Smoke checks and usage checklist

In [ ]:
# Lightweight checks that do not call OpenAI or create embeddings.
assert settings.chunk_size > 0
assert 0 <= settings.chunk_overlap < settings.chunk_size
assert DocumentProcessor.SUPPORTED_EXTENSIONS == {".pdf", ".txt", ".md"}
assert RAGEngine._format_sources([]) == ("", [])
print("Notebook definitions and basic checks passed.")